# Econometría II — Tarea 1

Versión con código simple y directo.

## Enunciado

Para responder este conjunto de preguntas se debe utilizar la base de datos NLS. Esta contiene las siguientes variables:

- `lwage`: logaritmo del salario.
- `educ`: años de educación.
- `exper`: años de experiencia.
- `age`: edad en años.
- `feduc`: años de educación del padre.
- `meduc`: años de educación de la madre.
- `kww`: puntuación obtenida en una prueba de conocimientos.
- `iq`: puntuación de coeficiente intelectual.
- `black`: variable indicadora de raza.

### Preguntas

1. Estime un modelo de regresión lineal para el logaritmo del salario semanal en función de la educación, la experiencia y la experiencia al cuadrado. Reporte las estimaciones de los coeficientes y sus errores estándar.
2. Prediga el efecto sobre el logaritmo del ingreso promedio de aumentar en un año el nivel educativo de todos los individuos.
3. ¿Es posible obtener el efecto anterior mediante un análisis de regresión que utilice un conjunto redefinido de covariables? Explique cómo hacerlo y estime el modelo correspondiente.
4. Prediga el efecto sobre el nivel promedio de ingresos de la siguiente política: aumentar a 12 años el nivel de educación de todos los individuos que actualmente tienen menos de 12 años de educación y mantener sin cambios el nivel educativo de los demás individuos.
5. Calcule el error estándar del efecto estimado para la política descrita en la pregunta anterior.

## Cargar la base

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

ruta = Path('data/NLS80V2.dta')
if not ruta.exists():
    from google.colab import files
    files.upload()
    ruta = Path('NLS80V2.dta')

datos = pd.read_stata(ruta)
datos['exper2'] = datos['exper'] ** 2
print('Observaciones:', len(datos))

Observaciones: 935


## 1. Regresión salarial

Estimamos `lwage` usando educación, experiencia y experiencia al cuadrado.

In [2]:
X = sm.add_constant(datos[['educ', 'exper', 'exper2']])
modelo = sm.OLS(datos['lwage'], X).fit()

resultado = pd.DataFrame({
    'coeficiente': modelo.params,
    'error estándar': modelo.bse
})
print(resultado.round(5))

        coeficiente  error estándar
const       5.05292         0.21333
educ        0.08383         0.00725
exper       0.06710         0.02394
exper2     -0.00158         0.00084


**Resultado:** el coeficiente de educación es **0,08383** y su error estándar es **0,00725**.

## 2. Aumentar un año la educación

A edad fija, un año más de educación significa un año menos de experiencia.

In [3]:
b = modelo.params
media_exper = datos['exper'].mean()
efecto2 = b['educ'] - b['exper'] + b['exper2'] * (1 - 2 * media_exper)

print('Efecto en log salario:', round(efecto2, 5))
print('Efecto porcentual:', round(100 * (np.exp(efecto2) - 1), 2), '%')

Efecto en log salario: 0.05822
Efecto porcentual: 6.0 %


**Resultado:** el log salario aumenta **0,05822**, aproximadamente **6,00 %**.

## 3. Regresión con variables redefinidas

In [4]:
a = 1 - 2 * media_exper
datos['z1'] = datos['educ'] + datos['exper']
datos['z2'] = datos['exper2'] - a * datos['educ']

X2 = sm.add_constant(datos[['educ', 'z1', 'z2']])
modelo2 = sm.OLS(datos['lwage'], X2).fit()

print('Efecto:', round(modelo2.params['educ'], 5))
print('Error estándar:', round(modelo2.bse['educ'], 5))

Efecto: 0.05822
Error estándar: 0.00596


**Resultado:** obtenemos nuevamente **0,05822**, con error estándar **0,00596**.

## 4. Política de educación mínima de 12 años

In [5]:
aumento = (12 - datos['educ']).clip(lower=0)
exper_nueva = datos['exper'] - aumento

cambio = (
    b['educ'] * aumento
    - b['exper'] * aumento
    + b['exper2'] * (exper_nueva ** 2 - datos['exper2'])
)

salario_nuevo = datos['wage'] * np.exp(cambio)
efecto4 = (salario_nuevo - datos['wage']).mean()

print('Personas afectadas:', (aumento > 0).sum())
print('Efecto promedio:', round(efecto4, 2))

Personas afectadas: 88
Efecto promedio: 8.98


**Resultado:** la política afecta a **88 personas** y aumenta el salario promedio en **8,98 unidades**.

## 5. Error estándar mediante bootstrap

In [6]:
def efecto_politica(muestra):
    muestra = muestra.copy()
    muestra['exper2'] = muestra['exper'] ** 2
    X = sm.add_constant(muestra[['educ', 'exper', 'exper2']])
    b = sm.OLS(muestra['lwage'], X).fit().params
    aumento = (12 - muestra['educ']).clip(lower=0)
    exper_nueva = muestra['exper'] - aumento
    cambio = b['educ']*aumento - b['exper']*aumento + b['exper2']*(exper_nueva**2 - muestra['exper2'])
    return (muestra['wage'] * (np.exp(cambio) - 1)).mean()

np.random.seed(123)
efectos = [efecto_politica(datos.sample(len(datos), replace=True)) for _ in range(1000)]

print('Error estándar:', round(np.std(efectos, ddof=1), 2))
print('Intervalo 95%:', np.round(np.percentile(efectos, [2.5, 97.5]), 2))

Error estándar: 1.31
Intervalo 95%: [ 6.59 11.57]


**Resultado:** el error estándar bootstrap es **1,31** y el intervalo al 95 % es aproximadamente **[6,59; 11,57]**.

> Los resultados son predicciones del modelo MCO; su interpretación causal requiere supuestos adicionales.